Simulation has two interfaces: a trial procedure advances a model and
generates item codes; the dataset wrapper assigns parameters, repeats trials
and returns a `RecallDataset`. Understanding that boundary makes it possible
to supply a different trial procedure while retaining dataset handling.

## The trial procedure is an entry point

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class TrialSimulator(Protocol):
    def __call__(
        self,
        model: MemorySearch,
        present: Integer[Array, " study_events"],
        trial: Integer[Array, " recalls"],
        rng: PRNGKeyArray,
    ) -> tuple[MemorySearch, Integer[Array, " recall_events"]]: ...
```

The callback receives a fresh model, its study sequence, a recall sequence
and a JAX random key. It returns the updated model and generated within-list
item codes. `present` and `trial` use the same item coding at this boundary;
the dataset adapter translates recorded study positions beforehand.

A compatible function is supplied through **`simulate_trial_fn`** to
`MemorySearchSimulator` or `simulate_h5_from_h5()`. It must preserve the
required array shapes and work with the JAX transformations used to run
multiple trials. Its use of the supplied recall sequence determines whether
the simulation conditions on observed behavior.

| Existing trial procedure | Use of the recorded sequence |
|---|---|
| `simulate_study_and_free_recall` | Generates recall using the model's stopping policy, up to the study-width output limit. |
| `simulate_study_first_recall_and_free_recall` | Supplies the recorded first recall, then simulates subsequent responses. |
| `simulate_study_free_recall_and_forced_stop` | Masks generated responses using the observed nonzero response slots. |

The forced-stop procedure, paired with a no-stop model, conditions on the
recorded recall length. A matching recall count is imposed, not predicted.
Its returned model has processed the sampled sequence before output masking;
masking the returned array is not itself a model state update.

## The dataset wrapper

`simulate_h5_from_h5()` accepts and returns in-memory dictionaries despite
its name. HDF5 storage is a separate step. Its central arguments are:

| Argument | Meaning |
|---|---|
| `model_factory_cls`, `dataset`, `features` | Model construction and experimental inputs. |
| `parameters` | One-dimensional parameter arrays aligned with a `subject` ID array. |
| `trial_mask` | Selected rows of the observation dataset. |
| `experiment_count` | Replications of each selected trial. |
| `rng` | JAX random key split across generated trials. |
| `simulate_trial_fn` | The compatible trial callback above. |

Participant IDs need not be consecutive or equal row indices. The wrapper
matches each trial's ID to `parameters['subject']`, then uses the same entry
from every parameter array. To apply one pooled fit to several participants,
repeat its parameter values and supply those participants' actual IDs.

In [ ]:
#| code-summary: "0 — Locate the project inputs"

from pathlib import Path

from jaxcmr.helpers import find_project_root, resolve_project_path

try:
    root_path = Path(find_project_root())
except FileNotFoundError:
    root_path = Path.cwd()


In [1]:
#| code-summary: "Assign pooled parameters to two participants and supply the trial callback"

from pathlib import Path
import json
import jax
import jax.numpy as jnp
from jaxcmr.helpers import load_data, generate_trial_mask
from jaxcmr.components import context, linear_memory
from jaxcmr.components.termination import NoStopTermination
from jaxcmr.models.cmr import make_factory

jax.config.update("jax_enable_x64", True)
data = load_data(resolve_project_path("data/Dupertuys2026.h5", root_path))
record = json.loads(Path(resolve_project_path("data/fits/Dupertuys2026_CMR_demo.json", root_path)).read_text())
parameters = {name: values[0] for name, values in record["fits"].items()
              if name != "subject"}
for name in ("allow_repeated_recalls", "learn_after_context_update"):
    parameters[name] = bool(parameters[name])

CMRFactory = make_factory(
    mfc_create_fn=linear_memory.init_mfc,
    mcf_create_fn=linear_memory.init_mcf,
    context_create_fn=context.init,
    termination_policy_create_fn=NoStopTermination,
)

from jax import random
from jaxcmr.simulation import (
    simulate_h5_from_h5, simulate_study_free_recall_and_forced_stop,
)

mixed = generate_trial_mask(data, "data['list_type'] == 1")
participant_ids = jnp.unique(data["subject"][mixed])[:2]
trial_mask = mixed & jnp.isin(data["subject"].ravel(), participant_ids)
simulation_parameters = {
    name: jnp.full(participant_ids.size, value)
    for name, value in parameters.items()
}
simulation_parameters["subject"] = participant_ids

simulated = simulate_h5_from_h5(
    model_factory_cls=CMRFactory, dataset=data, features=None,
    parameters=simulation_parameters, trial_mask=trial_mask,
    experiment_count=2, rng=random.PRNGKey(0),
    simulate_trial_fn=simulate_study_free_recall_and_forced_stop,
)
print("Parameter-row IDs:", simulation_parameters["subject"])
print("Selected trials:", int(trial_mask.sum()))
print("Generated participant IDs:", simulated["subject"].ravel())
print("Generated recall shape:", simulated["recalls"].shape)

Parameter-row IDs: [1 2]
Selected trials: 4
Generated participant IDs: [1 1 1 1 2 2 2 2]
Generated recall shape: (8, 12)


This illustration limits the mixed-list selection to two participants and
creates two replications per list. Replicating their trials does not create
additional participants. Reusing a key with the same computation reproduces
its draws; different simulation runs should record their seeds and settings.

## The returned dataset

The wrapper repeats study information and trial conditions with each list.
It converts generated item codes back to study-position recall codes, using
the first occurrence when a study item repeats. The returned dictionary can
therefore enter dataset-based [recall analyses](analyses.ipynb).

In [2]:
#| code-summary: "Inspect the dataset boundary after simulation"

print("First two generated recall sequences:")
print(simulated["recalls"][:2])
print("Presentations retained:", bool(jnp.array_equal(
    simulated["pres_itemnos"],
    jnp.repeat(data["pres_itemnos"][trial_mask], 2, axis=0),
)))
print("Omitted response fields:", sorted(set(data) - set(simulated)))

First two generated recall sequences:
[[10  5  8 11  4  2  0  0  0  0  0  0]
 [ 3  1  2  8  7 10  0  0  0  0  0  0]]
Presentations retained: True
Omitted response fields: ['rec_itemids']


Original response measurements cannot describe new responses. The wrapper
omits `rec_itemids`, `irt`, `cue_clips` and `_should_tabulate` when present.
Supply additional response-dependent field names through
`recall_dependent_fields`. Study-aligned fields remain attached to their lists.

Use [Data](data.ipynb) for coding and storage conventions. The
[simulation template](../templates/simulation.ipynb) handles a complete run,
saved outputs and behavioral comparisons; the [parameter exploration
template](../templates/parameter_shifting.ipynb) repeats that procedure under
deliberately changed parameters.